In [2]:
!pip install --upgrade googletrans transformers torch

  Using cached googletrans-4.0.2-py3-none-any.whl.metadata (10 kB)
Using cached googletrans-4.0.2-py3-none-any.whl (18 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/124.1 MB 4.8 MB/s eta 0:00:26
   --- ------------------------------------ 11.3/124.1 MB 33.6 MB/s eta 0:00:04
   --- ------------------------------------ 12.3/124.1 MB 36.8 MB/s eta 0:00:04
   --- ------------------------------------ 12.3/124.1 MB 36.8 MB/s eta 0:00:04
   ---- ----------------------------------- 12.6/124.1 MB 14.3 MB/s eta 0:00:08
   ---- ----------------------------------- 13.4/124.1 MB 11.8 MB/s eta 0:00:10
   ---- ----------------------------------- 14.2/124.1 MB 10.6 MB/s eta 0:00:11
   ---- ----------------------------------- 15.2/124.1 MB 9.9 MB/s eta 0:00:12
   ----- ---------------------------------- 17.0/124.1 MB 9.8 MB/s eta 0:00:11
   ------ --------------------------------- 19.9/124.1 MB 10.2 MB/s eta 0:00:11
   -------

In [3]:
from transformers import pipeline
import pandas as pd

In [4]:
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [5]:
sentence = "Barack Obama was born in Hawaii."

results = ner_pipeline(sentence)

results

[{'entity_group': 'PER',
  'score': np.float32(0.9992919),
  'word': 'Barack Obama',
  'start': 0,
  'end': 12},
 {'entity_group': 'LOC',
  'score': np.float32(0.9997441),
  'word': 'Hawaii',
  'start': 25,
  'end': 31}]

In [6]:
def display_entities(sentence):
    results = ner_pipeline(sentence)

    print("Sentence:", sentence)
    print("-" * 60)
    print(f"{'Entity':<25} {'Label':<15} {'Score':<10}")
    print("-" * 60)

    for entity in results:
        print(
            f"{entity['word']:<25} "
            f"{entity['entity_group']:<15} "
            f"{entity['score']:.2f}"
        )

In [7]:
display_entities("Barack Obama was born in Hawaii.")

Sentence: Barack Obama was born in Hawaii.
------------------------------------------------------------
Entity                    Label           Score     
------------------------------------------------------------
Barack Obama              PER             1.00
Hawaii                    LOC             1.00


In [8]:
sentences = [
    "Barack Obama was born in Hawaii.",
    "Elon Musk founded SpaceX.",
    "Bill Gates co-founded Microsoft.",
    "Apple released a new iPhone in California.",
    "Google has an office in Singapore.",
    "Microsoft opened a new research center in London.",
    "Mark Zuckerberg founded Facebook in 2004.",
    "Amazon was founded by Jeff Bezos.",
    "The United Nations held a meeting in New York.",
    "Tesla opened a factory in Berlin.",
    "Dr. Aung San Suu Kyi visited Yangon.",
    "Samsung announced a new smartphone in South Korea.",
    "The Olympic Games were held in Paris.",
    "NASA launched a spacecraft from Florida.",
    "Albert Einstein was born in Germany.",
    "The World Health Organization held a conference in Geneva.",
    "Apple and Google announced a technology partnership.",
    "Cristiano Ronaldo played football in Madrid.",
    "The Nobel Prize ceremony took place in Stockholm.",
    "Myanmar celebrated Independence Day on January 4, 2026."
]

In [9]:
for i, sentence in enumerate(sentences, 1):
    print(f"\nSentence {i}:")
    display_entities(sentence)


Sentence 1:
Sentence: Barack Obama was born in Hawaii.
------------------------------------------------------------
Entity                    Label           Score     
------------------------------------------------------------
Barack Obama              PER             1.00
Hawaii                    LOC             1.00

Sentence 2:
Sentence: Elon Musk founded SpaceX.
------------------------------------------------------------
Entity                    Label           Score     
------------------------------------------------------------
El                        PER             0.53
##on                      ORG             0.60
Mu                        PER             0.87
##sk                      ORG             0.47
SpaceX                    ORG             1.00

Sentence 3:
Sentence: Bill Gates co-founded Microsoft.
------------------------------------------------------------
Entity                    Label           Score     
----------------------------------------------

In [10]:
all_results = []

for i, sentence in enumerate(sentences, 1):

    results = ner_pipeline(sentence)

    for entity in results:
        all_results.append({
            "Sentence": i,
            "Entity": entity["word"],
            "Label": entity["entity_group"],
            "Score": round(float(entity["score"]), 2)
        })

results_df = pd.DataFrame(all_results)

results_df

,Sentence,Entity,Label,Score
0,1,Barack Obama,PER,1.00
1,1,Hawaii,LOC,1.00
2,2,El,PER,0.53
3,2,##on,ORG,0.60
4,2,Mu,PER,0.87
5,2,##sk,ORG,0.47
6,2,SpaceX,ORG,1.00
7,3,Bill Gates,PER,1.00
8,3,Microsoft,ORG,1.00
9,4,Apple,ORG,1.00


In [11]:
import re

date_pattern = r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b'

In [12]:
text = "Myanmar celebrated Independence Day on January 4, 2026."

dates = re.findall(date_pattern, text)

print(dates)

['January 4, 2026']


In [14]:
import re

date_pattern = r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b'


def display_entities_with_date(sentence):

    results = ner_pipeline(sentence)

    print("Sentence:", sentence)
    print("-" * 65)
    print(f"{'Entity':<30} {'Label':<15} {'Score':<10}")
    print("-" * 65)

    # BERT entities
    for entity in results:
        print(
            f"{entity['word']:<30} "
            f"{entity['entity_group']:<15} "
            f"{entity['score']:.2f}"
        )

    # DATE entities
    dates = re.findall(date_pattern, sentence)

    for date in dates:
        print(
            f"{date:<30} "
            f"{'DATE':<15} "
            f"{1.00:<10.2f}"
        )

### 1. Why is BERT called a contextualized language model?

BERT is called a contextualized language model because it understands the meaning of a word based on the words surrounding it. The same word can have different meanings in different sentences, and BERT uses the complete context to understand it. For example, the word “bank” has different meanings in “I deposited money in the bank” and “I sat near the river bank.” BERT can represent these words differently based on their context.


### 2. Difference between Word2Vec and BERT

| Word2Vec                                                                   | BERT                                                                                |
| -------------------------------------------------------------------------- | ----------------------------------------------------------------------------------- |
| Produces a fixed vector for each word.                                     | Produces contextual representations of words.                                       |
| The same word normally has the same representation in different sentences. | The representation changes according to the surrounding words.                      |
| Uses local word relationships to learn word embeddings.                    | Uses Transformer architecture and attention to understand context.                  |
| Cannot effectively distinguish different meanings of the same word.        | Can distinguish different meanings of the same word based on context.               |
| Mainly used for word embeddings.                                           | Can be fine-tuned for NER, classification, question answering, and other NLP tasks. |

**In short:** Word2Vec gives mostly static word representations, while BERT gives context-dependent representations.
